In [2]:
import torch
import networkx as nx
import time
from torch_geometric.utils import barabasi_albert_graph, to_networkx
from torch_geometric.data import Data

/home/ankit/fraud_detection/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def large_scale_directional_blindness():
    num_nodes = 10000
    num_edges_per_node = 3
    
    # Generate a scale-free graph (similar to crypto transaction networks)
    edge_index = barabasi_albert_graph(num_nodes, num_edges_per_node)
    
    # Randomly assign directions to simulate money flow
    # (Barabasi-Albert is undirected by default in PyG)
    mask = torch.rand(edge_index.size(1)) > 0.5
    edge_index[0, mask], edge_index[1, mask] = edge_index[1, mask].clone(), edge_index[0, mask].clone()
    
    data = Data(edge_index=edge_index, num_nodes=num_nodes)
    
    print(f"Graph generated with {num_nodes} nodes and {edge_index.size(1)} directed edges.\n")
    
    # 1. The Standard GSN Approach (Undirected Triangles)
    start = time.time()
    g_undirected = to_networkx(data, to_undirected=True)
    undirected_triangles = sum(nx.triangles(g_undirected).values()) // 3
    print(f"Standard GSN 'Triangle' Count: {undirected_triangles} (Took {time.time() - start:.2f}s)")
    
    # 2. The Directed Reality (Money Laundering Cycles)
    start = time.time()
    g_directed = to_networkx(data, to_undirected=False)
    directed_cycles = len(list(nx.simple_cycles(g_directed, length_bound=3)))
    print(f"Actual Directed Laundering Cycles: {directed_cycles} (Took {time.time() - start:.2f}s)")

In [4]:
if __name__ == "__main__":
    large_scale_directional_blindness()

Graph generated with 10000 nodes and 59870 directed edges.

Standard GSN 'Triangle' Count: 511 (Took 0.52s)
Actual Directed Laundering Cycles: 15173 (Took 938.47s)
